In [1]:
import numpy as np
import time
from datetime import datetime
import json

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm
from pyg_dataset import NetlistDataset

In [8]:
dataset = NetlistDataset(data_dir="../data/superblue", load_pd = True, load_pe = True, pl = True, processed = True, load_indices=None)

100%|██████████| 12/12 [00:02<00:00,  4.78it/s]


In [9]:
node_features = []
node_demand = []
node_congestion = []
for i in [0, 1, 2, 3, 6, 7, 8, 9, 10, 11]:
    node_features.append(dataset[i].node_features.cpu().numpy())
    node_demand.append(dataset[i].node_demand.cpu().numpy())
    node_congestion.append(dataset[i].node_congestion.cpu().numpy())

train_features = np.concatenate(node_features)
train_demand = np.concatenate(node_demand)
train_congestion = np.concatenate(node_congestion)

test_features = dataset[5].node_features.cpu().numpy()
test_demand = dataset[5].node_demand.cpu().numpy()
test_congestion = dataset[5].node_congestion.cpu().numpy()

# idx = np.random.choice(a=[True, False], p=[0.2, 0.8], size=train_features.shape[0])
# train_features = train_features[idx]
# train_congestion = train_congestion[idx]

In [10]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_features, hidden_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, hidden_dim),
            nn.ReLU(),
            # nn.Linear(hidden_dim, hidden_dim),
            # nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, X):
        return self.layers(X)

In [21]:
num_features = train_features.shape[1]
hidden_dim = 16
device = 'cuda'
epochs = 10
learning_rate = 0.001
# batch_size = 9_000_000
batch_size = 512_000

In [22]:
torch.cuda.empty_cache()
X_train = torch.Tensor(train_features).to(device)
y_train = torch.Tensor(train_demand.reshape(-1, 1)).to(device)
X_test = torch.Tensor(test_features).to(device)
y_test = torch.Tensor(test_demand.reshape(-1, 1)).to(device)

dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
num_batches = len(dataloader)

In [23]:
# model = NeuralNetwork(num_features, hidden_dim).to(device)
model = nn.Sequential(
    nn.Linear(45, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate,  weight_decay=0.01)

In [24]:
start_time = time.time()
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_X, batch_y in tqdm(dataloader):
        # batch_X = batch_X.to(device)
        # batch_y = batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y.float()) 
        train_loss += loss.item()
        loss.backward()
        optimizer.step()

    model.eval()
    y_pred = model(X_test)
    mse = criterion(y_pred, y_test)
    train_loss /= num_batches
    valid_loss = mse.item()
    cur_time = time.time() - start_time
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}, Time: {cur_time:.2f}")

100%|██████████| 18/18 [06:13<00:00, 20.74s/it]


Epoch [1/10], Train Loss: 719.2887, Valid Loss: 294.0227, Time: 373.31


 17%|█▋        | 3/18 [01:17<06:25, 25.72s/it]


KeyboardInterrupt: 